**ResNet** (short for **Residual Network**) is a Convolutional Neural Network architecture introduced by Kaiming He and colleagues at Microsoft Research in 2015. It solved the **degradation problem** in very deep neural networks by adding **skip connections** (shortcut connections) that let layers learn residual functions rather than complete mappings from scratch.

---

### Core Idea: Residual Learning & Skip Connections

In plain deep networks (like stacked convolutional layers in VGGNet), adding more layers beyond a certain point can actually increase training and test error—not just from overfitting, but because optimization becomes harder and gradients diminish as they backpropagate through many layers.

ResNet changes a block's target mapping from $H(x)$ to:

$$F(x) + x$$

* **$x$**: The block input carried forward along an identity **skip connection**.
* **$F(x)$**: The **residual mapping** learned by a small stack of convolutional layers (typically two 3×3 convolutions for basic blocks, or a 1×1 → 3×3 → 1×1 bottleneck for deeper variants).
* **Addition ($F(x) + x$)**: Element-wise summation before or after the activation function. If an extra layer is not needed, the network can push $F(x)$ toward zero so the block acts close to an identity mapping without starving gradient flow.

When the input $x$ and output $F(x)$ differ in channel count or spatial resolution (such as at stride-2 downsampling stages), ResNet projects $x$ using a 1×1 convolution with matching stride so the shapes align for addition.

---

### Common ResNet Variants & Structure

Most ResNet models start with a stem convolution and pooling layer, pass through four stages of residual blocks (doubling channels at each stage while halving spatial dimensions), and finish with **global average pooling** and a fully connected classifier—avoiding the massive dense layers found in AlexNet and VGGNet.

| Variant | Layer Depth | Block Structure | Typical Use Case |
| --- | --- | --- | --- |
| **ResNet-18 / ResNet-34** | 18 or 34 layers | BasicBlock (two 3×3 convs + skip) | Lightweight vision tasks, fast baselines, edge/custom training |
| **ResNet-50 / ResNet-101 / ResNet-152** | 50, 101, or 152 layers | Bottleneck (1×1 reduce, 3×3 conv, 1×1 expand + skip) | High-capacity classification, detection, and segmentation backbones |

---

### Why ResNet Was a Turning Point

* **Trainable Very Deep Networks:** Enabled networks with dozens or hundreds of layers to train reliably without accuracy degrading as depth increased.
* **Leaner Classifier Head:** Replaced huge fully connected layers with global average pooling, cutting parameter bloat compared to VGGNet while improving generalization.
* **Foundational Backbone:** Residual skip connections became a standard pattern across computer vision backbones, feature pyramid networks, and even modern sequence and transformer architectures.

To write a custom **Residual Block** in PyTorch from scratch, subclass `nn.Module`, compute a residual branch $F(x)$ through convolutional and normalization layers, add the input $x$ via a skip connection, and apply the final activation.

---

### 1. Basic Residual Block (`BasicBlock`)

Used in architectures like **ResNet-18** and **ResNet-34**, a basic block stacks two 3×3 convolutions with Batch Normalization. When the channel count changes or `stride` is greater than 1 (downsampling spatial dimensions), a `1x1` convolution with matching `stride` projects the shortcut path $x$ so its tensor shape matches $F(x)$.

In [1]:
import torch
import torch.nn as nn


class BasicResidualBlock(nn.Module):
  expansion = 1

  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()

    # Main path F(x)
    self.conv1 = nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)

    self.conv2 = nn.Conv2d(
        out_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False,
    )
    self.bn2 = nn.BatchNorm2d(out_channels)

    # Shortcut path for x (identity or projection when shape changes)
    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(
              in_channels, out_channels, kernel_size=1, stride=stride, bias=False
          ),
          nn.BatchNorm2d(out_channels),
      )
    else:
      self.shortcut = nn.Identity()

  def forward(self, x):
    identity = self.shortcut(x)

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    out = out + identity
    out = self.relu(out)
    return out

### 2. Bottleneck Residual Block (`Bottleneck`)

Deeper models like **ResNet-50**, **ResNet-101**, and **ResNet-152** use a three-layer bottleneck (`1x1` channel reduction, `3x3` spatial convolution, `1x1` channel expansion by `expansion = 4`) to lower compute cost at high channel widths.

In [2]:
class BottleneckResidualBlock(nn.Module):
  expansion = 4

  def __init__(self, in_channels, bottleneck_channels, stride=1):
    super().__init__()
    out_channels = bottleneck_channels * self.expansion

    self.conv1 = nn.Conv2d(
        in_channels, bottleneck_channels, kernel_size=1, bias=False
    )
    self.bn1 = nn.BatchNorm2d(bottleneck_channels)

    self.conv2 = nn.Conv2d(
        bottleneck_channels,
        bottleneck_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )
    self.bn2 = nn.BatchNorm2d(bottleneck_channels)

    self.conv3 = nn.Conv2d(
        bottleneck_channels, out_channels, kernel_size=1, bias=False
    )
    self.bn3 = nn.BatchNorm2d(out_channels)

    self.relu = nn.ReLU(inplace=True)

    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(
              in_channels, out_channels, kernel_size=1, stride=stride, bias=False
          ),
          nn.BatchNorm2d(out_channels),
      )
    else:
      self.shortcut = nn.Identity()

  def forward(self, x):
    identity = self.shortcut(x)

    out = self.relu(self.bn1(self.conv1(x)))
    out = self.relu(self.bn2(self.conv2(out)))
    out = self.bn3(self.conv3(out))

    out = self.relu(out + identity)
    return out

### 3. Quick Shape & Gradient Check

Run a dummy forward and backward pass to confirm that dimensions match and gradients flow cleanly through both the residual branch and the skip connection:

In [3]:
x = torch.randn(8, 64, 32, 32, requires_grad=True)

basic_block = BasicResidualBlock(in_channels=64, out_channels=128, stride=2)
out_basic = basic_block(x)
out_basic.sum().backward()

print(
    "BasicBlock output shape:", out_basic.shape
)  # [8, 128, 16, 16]
print("Input gradient exists:", x.grad is not None)

x.grad.zero_()
bottleneck = BottleneckResidualBlock(
    in_channels=64, bottleneck_channels=32, stride=1
)
out_bottleneck = bottleneck(x)
out_bottleneck.sum().backward()

print(
    "Bottleneck output shape:", out_bottleneck.shape
)  # [8, 128, 32, 32]



BasicBlock output shape: torch.Size([8, 128, 16, 16])
Input gradient exists: True
Bottleneck output shape: torch.Size([8, 128, 32, 32])


### Key Implementation Details

* **Omit `bias=True` Before `BatchNorm2d`:** Batch normalization subtracts the channel mean, making a preceding convolution's bias redundant.
* **Match Stride on Downsampling:** When `stride=2` is set on `conv1` (or `conv2`), the `1x1` projection in `self.shortcut` must also use `stride=2` so height and width match during `out + identity`.
* **In-Place ReLU Caution:** Using `inplace=True` is fine for standard sequential activations, but avoid mutating a tensor in place if it is still referenced elsewhere in the computation graph.